# Reviewer Notebook: SO(k)-Invariant Gauge Classification (GS-002)
## Rigorous Cohomological Duality, Operator Norms, and Adversarial Verification
**Canonical Identifier:** `GS-002` (`so_k_invariant_gauge_classification`)  
**Mathematical Scope:** Finite graph cochain complexes $C^0 \xrightarrow{d_0} C^1 \xrightarrow{d_1} C^2$ with $\mathbb{R}^k$ vector stalks under declared $\mathrm{SO}(k)$ local symmetry.

---

### Abstract and Reviewer Objectives
This interactive notebook provides the full verification suite for GS-002:
1. **Local Stalk Rigidity (Theorem 3.1):** Rigorous verification of $\mathrm{SO}(k)$ transitivity on spheres forcing Euclidean local norms, and testing failure modes under discrete subgroup symmetries (e.g. signed permutations $B_k$).
2. **Mixed Vector Obstruction Duality (Theorem 4.2):** Second-Order Cone formulation of $\Phi_{2,1,c}([\omega])$ and dual circulation witness certificate verification (divergence-free constraint $d_0^* \varphi = 0$, $\|\varphi_e\|_2 \le c_e$, duality gap $< 10^{-5}$).
3. **Finite Anomaly Separation (Theorem 5.2):** Exact algebraic evaluation of coordinate-separable $\ell^1$ vs $\mathrm{SO}(k)$ mixed geometry on 3-cycles and general $n$-cycles.
4. **Mixed 2-Cell Closure Bound (Theorem 6.1):** Numerical validation of $\Gamma_{2,1,u}([h]) \le \|d_1\|_{(1,2,c)\to(1,2,u)} \Phi_{2,1,c}([h])$ and constructive saturation of the sharp operator norm.
5. **Invasive Fault-Injection & Recovery:** Injecting non-divergence perturbations, verifying detection, and recovering feasibility via Laplacian projection.

The examples below use floating-point arithmetic and explicit assertions. Random audits cover sampled inputs; numerical flags are tolerance-qualified. Read the [study guide](../docs/study_guide.md), attempt the [Landau quiz](../docs/landau_quiz.md), and consult the [publication review](../docs/publication_review.md) for fresh-build and kernel evidence.


In [1]:
import sys
import math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

def _boot_libs(*names):
    """Robustly resolve and import library directories regardless of CWD."""
    cwd = Path.cwd().resolve()
    search_dirs = [cwd] + list(cwd.parents)
    for ancestor in [cwd] + list(cwd.parents):
        cand = ancestor / "coda_atlas" / "papers" / "projection_obstruction" / "so_k_invariant_gauge_classification"
        if cand.exists():
            search_dirs.insert(0, cand)
            break
    for name in names:
        for base in search_dirs:
            p = (base / name).resolve()
            if p.is_dir() and p.exists():
                if str(p) not in sys.path:
                    sys.path.insert(0, str(p))
                return str(p)
            sub = base / "so_k_invariant_gauge_classification" / name
            if sub.is_dir() and sub.exists():
                sub_res = sub.resolve()
                if str(sub_res) not in sys.path:
                    sys.path.insert(0, str(sub_res))
                return str(sub_res)
    raise FileNotFoundError(f"none of {names} found from {cwd}")

code_dir = _boot_libs("code")

from so_k_gauge import (
    GraphComplex,
    Cochain,
    EuclideanGauge,
    CoordinateL1Gauge,
    LpGauge,
    AnisotropicGauge,
    audit_so_k_invariance,
    audit_sphere_constancy,
    MixedNormSolver,
    CoordinateL1Solver,
    analyze_triangle_separation,
    analyze_dimension_scaling,
    ClosureBoundEvaluator,
    InvasiveAdversary
)
print('Successfully loaded so_k_gauge mathematical suite!')

Successfully loaded so_k_gauge mathematical suite!


## 1. Theorem 3.1: Stalk Gauge Rigidity & Subgroup Counterexamples
Theorem 3.1 proves that any faithful, absolutely homogeneous gauge on $\mathbb{R}^k$ ($k \ge 2$) that is invariant under $\mathrm{SO}(k)$ is proportional to the Euclidean norm: $j(x) = c \|x\|_2$.

Below, we test invariance under continuous $\mathrm{SO}(k)$ rotations for:
- Euclidean gauge $j(x) = \|x\|_2$
- Coordinate gauge $j(x) = \sum |x_i|$
- $\ell^p$ gauge ($p=4$)
- Anisotropic gauge $j(x) = \|A x\|_2$

In [2]:
dim = 3
gauges = {
    'Euclidean L2 (SO(3)-invariant)': EuclideanGauge(1.0),
    'Coordinate L1 (Separable)': CoordinateL1Gauge(1.0),
    'Lp Gauge (p=4)': LpGauge(p=4.0),
    'Anisotropic Gauge': AnisotropicGauge(np.diag([1.5, 1.0, 0.8]))
}

print(f'=== Stalk Gauge SO({dim}) Invariance Audit ===')
for name, gauge in gauges.items():
    res = audit_so_k_invariance(gauge, dim=dim, num_samples=100)
    print(f"{name:32s} | Invariant: {res['is_so_k_invariant']!s:5s} | Max Deviation: {res['max_deviation']:.6f}")

audit_results = {name: audit_so_k_invariance(gauge, dim=dim, num_samples=100)
                 for name, gauge in gauges.items()}
assert audit_results['Euclidean L2 (SO(3)-invariant)']['is_so_k_invariant']
assert all(not r['is_so_k_invariant'] for name, r in audit_results.items()
           if name != 'Euclidean L2 (SO(3)-invariant)')
assert all(r['samples_tested'] == 100 for r in audit_results.values())

=== Stalk Gauge SO(3) Invariance Audit ===
Euclidean L2 (SO(3)-invariant)   | Invariant: True  | Max Deviation: 0.000000
Coordinate L1 (Separable)        | Invariant: False | Max Deviation: 0.986538
Lp Gauge (p=4)                   | Invariant: False | Max Deviation: 0.407675
Anisotropic Gauge                | Invariant: False | Max Deviation: 1.076669


## 2. Theorem 4.2: Mixed Vector Obstruction Duality via SOCP
We construct a finite triangle network $C^0 \xrightarrow{d_0} C^1 \xrightarrow{d_1} C^2$ and evaluate the analytical primal-dual duality:
$$\Phi_{2,1,c}([\omega]) = \max_{\substack{d_0^* \varphi = 0 \\ \|\varphi_e\|_2 \le c_e}} \sum_e \langle \varphi_e, \omega_e \rangle$$
with zero duality gap ($< 10^{-5}$).

In [3]:
# Triangle Complex (3 vertices, 3 edges, 1 face)
tri_complex = GraphComplex.create_cycle_graph(cycle_len=3, stalk_dim=2, edge_weights=[1.0, 1.0, 1.0])
w_data = np.array([
    [1.0, 0.0],  # Edge 0
    [0.0, 1.0],  # Edge 1
    [0.0, 0.0]   # Edge 2
])
omega = Cochain(w_data)

solver = MixedNormSolver(tri_complex)
cert = solver.solve(omega)

print('=== Primal-Dual Duality Certificate (Theorem 4.2) ===')
print(f'Primal Quotient Obstruction (Phi_2,1): {cert.primal_value:.8f} (Theoretical: {np.sqrt(2.0):.8f})')
print(f'Dual Circulation Value (Psi_2,1):     {cert.dual_value:.8f}')
print(f'Duality Gap |Phi - Psi|:               {cert.duality_gap:.8e}')
print(f'Max Divergence Norm ||d0^* phi||_max:  {cert.divergence_norm:.8e} (Must be 0)')
print(f'Max Dual Norm Violation:               {cert.max_dual_norm_violation:.8e} (Must be 0)')
print(f'Certificate Status:                    Optimal={cert.is_optimal}')

assert cert.is_optimal
assert cert.duality_gap <= solver.tol
assert cert.divergence_norm <= solver.tol
assert cert.max_dual_norm_violation <= solver.tol
assert math.isclose(cert.primal_value, math.sqrt(2), rel_tol=1e-12)
assert math.isclose(cert.dual_value, cert.primal_value, abs_tol=solver.tol)

=== Primal-Dual Duality Certificate (Theorem 4.2) ===
Primal Quotient Obstruction (Phi_2,1): 1.41421356 (Theoretical: 1.41421356)
Dual Circulation Value (Psi_2,1):     1.41421356
Duality Gap |Phi - Psi|:               2.22044605e-16
Max Divergence Norm ||d0^* phi||_max:  0.00000000e+00 (Must be 0)
Max Dual Norm Violation:               0.00000000e+00 (Must be 0)
Certificate Status:                    Optimal=True


## 3. Theorem 5.2: Finite Triangle Separation
Comparison of the exact quotient obstruction assigned to the same cycle class by:
- Coordinate-separable scalar observer ($\Phi_{1,1} = 2.0$)
- $\mathrm{SO}(2)$-invariant Euclidean observer ($\Phi_{2,1} = \sqrt{2} \approx 1.414$)

In [4]:
l1_solver = CoordinateL1Solver(tri_complex)
l1_res = l1_solver.solve(omega)

phi_11 = l1_res['primal_value']
phi_21 = cert.primal_value
ratio = phi_11 / phi_21

print('=== Finite Triangle Separation ===')
print(f'Coordinate L1 Quotient (Phi_1,1): {phi_11:.6f}')
print(f'SO(2) Mixed Quotient (Phi_2,1):    {phi_21:.6f}')
print(f'Discrepancy Ratio:                 {ratio:.6f} (Exact: {np.sqrt(2.0):.6f})')
print(f'Squared Ratio:                     {ratio**2:.6f} (Exact: 2.000000)')

assert math.isclose(phi_11, 2, abs_tol=1e-10)
assert math.isclose(phi_21, math.sqrt(2), rel_tol=1e-12)
assert math.isclose(ratio**2, 2, rel_tol=1e-12)

=== Finite Triangle Separation ===
Coordinate L1 Quotient (Phi_1,1): 2.000000
SO(2) Mixed Quotient (Phi_2,1):    1.414214
Discrepancy Ratio:                 1.414214 (Exact: 1.414214)
Squared Ratio:                     2.000000 (Exact: 2.000000)


## 4. Theorem 6.1: Mixed 2-Cell Closure Bound & Sharp Operator Norm
We evaluate the coboundary 2-cell failure metric $\Gamma_{2,1,u}([h]) = \|d_1 h\|_{1,2,u}$ and verify:
$$\Gamma_{2,1,u}([h]) \le \|d_1\|_{(1,2,c)\to(1,2,u)} \Phi_{2,1,c}([h])$$
where the mixed operator norm is constructively given by:
$$\|d_1\|_{(1,2,c)\to(1,2,u)} = \max_{e \in E} \frac{\sum_{f \in F} u_f |(d_1)_{f,e}|}{c_e}$$
We also construct the sharp achiever that saturates the inequality.

In [5]:
# Evaluate on Triangle
closure_eval = ClosureBoundEvaluator(tri_complex)
op_norm = closure_eval.compute_operator_norm()
res_closure = closure_eval.evaluate(omega)

print('=== Triangle Complex 2-Cell Closure Bound ===')
print(f'Operator Norm ||d1||:               {op_norm:.4f}')
print(f'2-Cell Failure Metric Gamma:        {res_closure.gamma_closure_defect:.6f}')
print(f'Quotient Obstruction Phi:           {res_closure.phi_quotient_obstruction:.6f}')
print(f'Closure Ratio R_cl = Gamma / Phi:   {res_closure.closure_ratio:.6f}')
print(f'Inequality Gamma <= ||d1|| * Phi:   {res_closure.is_bound_satisfied!s}')

# Evaluate on Theta Graph (multiple entangled 2-cells)
theta_complex = GraphComplex.create_theta_graph(stalk_dim=2)
theta_eval = ClosureBoundEvaluator(theta_complex)
theta_op_norm = theta_eval.compute_operator_norm()
achiever = theta_eval.construct_sharp_achiever()
res_achiever = theta_eval.evaluate(achiever)

print('\n=== Theta Graph (Multi-Face Complex) ===')
print(f'Theta Operator Norm ||d1||:         {theta_op_norm:.4f} (Edge 1 belongs to 2 faces)')
print(f'Achiever Bound Satisfied:           {res_achiever.is_bound_satisfied!s}')

assert op_norm == 1 and theta_op_norm == 2
assert res_closure.certified_bound_holds and res_achiever.certified_bound_holds
assert math.isclose(res_closure.closure_ratio, 1, rel_tol=1e-12)
assert res_closure.certified_quotient_value is not None

=== Triangle Complex 2-Cell Closure Bound ===
Operator Norm ||d1||:               1.0000
2-Cell Failure Metric Gamma:        1.414214
Quotient Obstruction Phi:           1.414214
Closure Ratio R_cl = Gamma / Phi:   1.000000
Inequality Gamma <= ||d1|| * Phi:   True

=== Theta Graph (Multi-Face Complex) ===
Theta Operator Norm ||d1||:         2.0000 (Edge 1 belongs to 2 faces)
Achiever Bound Satisfied:           True


## 5. Invasive Fault-Injection & Recovery Test
We deliberately inject a non-divergence-free perturbation into the dual circulation vector $\varphi$ and test the detection and recovery mechanisms.

In [6]:
adversary = InvasiveAdversary(tri_complex)

print('=== Invasive Test 1: Divergence Perturbation & Recovery ===')
div_result = adversary.test_divergence_injection_and_recovery(omega, perturbation_magnitude=1.0)
print(f"Fault:                  {div_result.fault_injected}")
print(f"Detected Failure:       {div_result.detected_failure}")
print(f"Corrupted Max Div:      {div_result.metrics['max_divergence_corrupted']:.4f}")
print(f"Recovered Max Div:      {div_result.metrics['max_divergence_recovered']:.6e}")
print(f"Recovery Successful:    {div_result.recovery_successful}")

print('\n=== Invasive Test 2: Gauge Anisotropy Injection ===')
aniso_result = adversary.test_gauge_anisotropy_injection(epsilon=0.25)
print(f"Fault:                  {aniso_result.fault_injected}")
print(f"Detected Failure:       {aniso_result.detected_failure}")
print(f"Corrupted Deviation:    {aniso_result.metrics['corrupted_max_deviation']:.4f}")
print(f"Restored Deviation:     {aniso_result.metrics['restored_max_deviation']:.6e}")
print(f"Recovery Successful:    {aniso_result.recovery_successful}")

print('\n=== Invasive Test 3: Lipschitz Noise Resilience ===')
noise_result = adversary.test_lipschitz_noise_resilience(omega, noise_level=0.08)
print(f"Fault:                  {noise_result.fault_injected}")
print(f"Absolute Difference:    {noise_result.metrics['absolute_difference']:.6f}")
print(f"Theoretical Bound:      {noise_result.metrics['theoretical_upper_bound']:.6f}")
print(f"Lipschitz Bound Holds:  {noise_result.recovery_successful}")

assert div_result.detected_failure and div_result.recovery_successful
assert aniso_result.detected_failure and aniso_result.recovery_successful
assert noise_result.recovery_successful

=== Invasive Test 1: Divergence Perturbation & Recovery ===
Fault:                  Additive non-divergence delta of magnitude 1.0
Detected Failure:       True
Corrupted Max Div:      1.0000
Recovered Max Div:      2.220446e-16
Recovery Successful:    True

=== Invasive Test 2: Gauge Anisotropy Injection ===
Fault:                  Diagonal anisotropy scaling with epsilon=0.25
Detected Failure:       True
Corrupted Deviation:    0.4513
Restored Deviation:     8.881784e-16
Recovery Successful:    True

=== Invasive Test 3: Lipschitz Noise Resilience ===
Fault:                  Gaussian noise with std=0.08, ||eta||=0.2755
Absolute Difference:    0.092479
Theoretical Bound:      0.275517
Lipschitz Bound Holds:  True


## 6. Formalization and evidence scope

| Claim | Included Lean evidence | Remaining step |
|---|---|---|
| Local rigidity | Native-norm radial collapse; separate planar circle rotation | General-k transitivity and full Euclidean group bridge |
| Cycle quotient | Native-norm cost and anchored zero-sum repair kernels | Ordinary graph-incidence identification in prose |
| Triangle separation | Rational squared-cost inequality | Geometric cost identification in prose |
| Mixed duality | Repair-invariant weak-duality kernels | Standard strong-duality equality and attainment |
| Closure | Abstract repair invariance and infimum step | Concrete operator-norm instantiation |

The retained Lean receipt is dated 2026-09-28. The publication review records
current source custody and distinguishes the fresh checks actually performed
from the Lean/Lake and notebook kernel gates still required for release.
Execution counts alone do not establish fresh kernel replay.


## 7. Signed weights and a small-circulation regression

Reversing a stored edge changes its data and traversal sign together. A tiny
coordinate circulation can still have a substantial weighted quotient. Both
checks below use an analytical cycle formula evaluated in floating point.


In [7]:
signed_graph = GraphComplex.create_cycle_graph(
    3, stalk_dim=2, orientations=[1, -1, 1], edge_weights=[3, 1, 2])
signed_data = Cochain([[1, 0], [0, -1], [0, 0]])
signed_cert = MixedNormSolver(signed_graph).solve(signed_data)
assert signed_cert.is_optimal
assert math.isclose(signed_cert.primal_value, math.sqrt(2), rel_tol=1e-12)

tiny_graph = GraphComplex.create_cycle_graph(3, stalk_dim=1, edge_weights=[1e12]*3)
tiny_cert = MixedNormSolver(tiny_graph).solve(Cochain([[1e-9], [0], [0]]))
assert tiny_cert.is_optimal and tiny_cert.solver_method == 'cycle_closed_form'
assert math.isclose(tiny_cert.primal_value, 1000, rel_tol=1e-12)
assert math.isclose(tiny_cert.dual_value, 1000, rel_tol=1e-12)
try:
    audit_so_k_invariance(lambda x: float('nan'), dim=2, num_samples=2)
except ValueError:
    pass
else:
    raise AssertionError('Nonfinite gauge outputs cannot pass an audit')
print('Signed weighted cycle and small-circulation assertions passed.')


Signed weighted cycle and small-circulation assertions passed.
